# RedSentinel — XSS Dataset Data Analytics

Comprehensive analysis, visualizations, and insights for the 59,122-sample XSS dataset.


## 1. Import Required Libraries

In [1]:
import os, re, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings("ignore")
matplotlib.use("Agg")   # headless — saves cleanly to disk

# ---------- paths ----------
ROOT   = Path("/home/moon/projects/ratopaleydai")
SPLITS = ROOT / "dataset" / "splits"
PROC   = ROOT / "dataset" / "processed"
CHARTS = ROOT / "docs" / "charts"
CHARTS.mkdir(parents=True, exist_ok=True)

# ---------- global style ----------
PALETTE = {
    "script_injection":  "#e74c3c",
    "event_handler":     "#e67e22",
    "js_uri":            "#f39c12",
    "tag_injection":     "#2ecc71",
    "template_injection":"#1abc9c",
    "dom_sink":          "#3498db",
    "attribute_escape":  "#9b59b6",
    "generic":           "#95a5a6",
}
SEV_PALETTE = {"high": "#e74c3c", "medium": "#f39c12", "low": "#2ecc71"}

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({"figure.dpi": 140, "font.size": 10, "axes.titlesize": 12})

print("Libraries loaded  ✓")
print(f"Charts will be saved to: {CHARTS}")


/tmp/ipykernel_36921/2472608138.py:6: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Libraries loaded  ✓
Charts will be saved to: /home/moon/projects/ratopaleydai/docs/charts


## 2. Load and Explore Data

In [2]:
# ── Load all three splits ──────────────────────────────────────────────────────
train = pd.read_csv(SPLITS / "train.csv")
val   = pd.read_csv(SPLITS / "val.csv")
test  = pd.read_csv(SPLITS / "test.csv")

for df, name in [(train, "train"), (val, "val"), (test, "test")]:
    df["split"] = name

full = pd.concat([train, val, test], ignore_index=True)

# Normalise
full["context"]  = full["context"].str.strip().str.lower()
full["severity"] = full["severity"].str.strip().str.lower()
full["source"]   = full["source"].str.strip().str.lower()
full["length"]   = pd.to_numeric(full["length"], errors="coerce")

print(f"Total samples : {len(full):,}")
print(f"Columns       : {list(full.columns)}")
print(f"\nSplit counts:")
print(full["split"].value_counts().to_string())
print(f"\nContext classes ({full['context'].nunique()}):")
print(full["context"].value_counts().to_string())
print(f"\nSeverity classes:")
print(full["severity"].value_counts().to_string())


Total samples : 59,122
Columns       : ['payload', 'context', 'technique', 'severity', 'length', 'source', 'split']

Split counts:
split
train    41385
test      8869
val       8868

Context classes (8):
context
tag_injection         23157
attribute_escape      16774
dom_sink               6365
event_handler          6027
script_injection       2535
js_uri                 2327
template_injection     1221
generic                 716

Severity classes:
severity
medium    39856
low       10087
high       9179


In [3]:
# ── Data types + nulls ────────────────────────────────────────────────────────
print("=== dtypes ===")
print(full.dtypes)
print("\n=== Null counts ===")
print(full.isnull().sum())
print("\n=== Payload length stats ===")
print(full["length"].describe().round(1).to_string())
print(f"\nShortest  : {full['length'].min():.0f} chars  → \"{full.loc[full['length'].idxmin(), 'payload'][:80]}\"")
print(f"Longest   : {full['length'].max():.0f} chars")
print(f"Mean      : {full['length'].mean():.1f} chars")
print(f"Median    : {full['length'].median():.0f} chars")


=== dtypes ===
payload      object
context      object
technique    object
severity     object
length        int64
source       object
split        object
dtype: object

=== Null counts ===
payload      0
context      0
technique    0
severity     0
length       0
source       0
split        0
dtype: int64

=== Payload length stats ===
count    59122.0
mean        60.9
std         34.6
min          6.0
25%         41.0
50%         54.0
75%         71.0
max       1504.0

Shortest  : 6 chars  → "&#x3c;"
Longest   : 1504 chars
Mean      : 60.9 chars
Median    : 54 chars


## 3. Data Cleaning & Preprocessing

In [4]:
# ── Drop nulls + duplicates ────────────────────────────────────────────────────
before = len(full)
full = full.dropna(subset=["payload", "context", "severity"])
full = full[full["payload"].str.strip() != ""]
full = full.drop_duplicates(subset=["payload"])
after = len(full)
print(f"Rows before cleaning : {before:,}")
print(f"Rows after  cleaning : {after:,}  (dropped {before-after:,})")

# ── Derived columns ────────────────────────────────────────────────────────────
full["payload_len_bucket"] = pd.cut(
    full["length"],
    bins=[0, 20, 40, 60, 80, 100, 150, 200, 300, 1000],
    labels=["0-20","21-40","41-60","61-80","81-100","101-150","151-200","201-300","301+"]
)

# Boolean flags
full["has_script_tag"]   = full["payload"].str.contains(r"<script", case=False, regex=True)
full["has_event_attr"]   = full["payload"].str.contains(r"\bon\w+\s*=", case=False, regex=True)
full["has_js_uri"]       = full["payload"].str.contains(r"javascript:", case=False, regex=True)
full["has_dom_func"]     = full["payload"].str.contains(
    r"innerHTML|document\.write|\.src\s*=|eval\(|setTimeout|setInterval", case=False, regex=True)
full["has_template"]     = full["payload"].str.contains(
    r"\{\{|\$\{|<%|#\{", case=False, regex=True)
full["has_data_uri"]     = full["payload"].str.contains(r"data:", case=False, regex=True)
full["has_obfuscation"]  = full["payload"].str.contains(
    r"&#x|&#\d|\\u00|%3c|fromCharCode|btoa|atob", case=False, regex=True)
full["has_cookie_exfil"] = full["payload"].str.contains(
    r"document\.cookie|localStorage|sessionStorage|navigator\.", case=False, regex=True)

print("\n=== Derived flag prevalence ===")
flag_cols = [c for c in full.columns if c.startswith("has_")]
print(full[flag_cols].sum().to_string())


Rows before cleaning : 59,122
Rows after  cleaning : 59,122  (dropped 0)

=== Derived flag prevalence ===
has_script_tag       2426
has_event_attr      41685
has_js_uri           3125
has_dom_func         2246
has_template         1016
has_data_uri          202
has_obfuscation     21669
has_cookie_exfil     2739


## 4. Exploratory Data Analysis (EDA)

In [5]:
# ── Per-class stats ────────────────────────────────────────────────────────────
ctx_stats = (
    full.groupby("context")["length"]
    .agg(count="count", mean="mean", median="median", std="std", min="min", max="max")
    .sort_values("count", ascending=False)
    .round(1)
)
total = ctx_stats["count"].sum()
ctx_stats["pct"] = (ctx_stats["count"] / total * 100).round(2)
print("=== Context class statistics ===")
print(ctx_stats.to_string())

print("\n=== Severity statistics ===")
sev_stats = (
    full.groupby("severity")["length"]
    .agg(count="count", mean="mean", median="median")
    .round(1)
)
sev_stats["pct"] = (sev_stats["count"] / total * 100).round(2)
print(sev_stats.to_string())

print("\n=== Source breakdown ===")
src_stats = full["source"].value_counts(normalize=False)
src_pct   = full["source"].value_counts(normalize=True).mul(100).round(2)
print(pd.DataFrame({"count": src_stats, "pct": src_pct}).to_string())

print("\n=== Technique breakdown (top 20) ===")
print(full["technique"].value_counts().head(20).to_string())


=== Context class statistics ===
                    count  mean  median   std  min   max    pct
context                                                        
tag_injection       23157  71.0    65.0  34.6   10   418  39.17
attribute_escape    16774  49.1    47.0  15.9   11   295  28.37
dom_sink             6365  73.9    68.0  35.9   14   736  10.77
event_handler        6027  44.9    42.0  17.4   12   283  10.19
script_injection     2535  58.1    52.0  60.0    8  1504   4.29
js_uri               2327  63.3    53.0  51.4   16  1210   3.94
template_injection   1221  41.4    33.0  46.3    9   567   2.07
generic               716  61.9    51.0  67.0    6   972   1.21

=== Severity statistics ===
          count  mean  median    pct
severity                            
high       9179  66.6    62.0  15.53
low       10087  63.5    55.0  17.06
medium    39856  58.9    50.0  67.41

=== Source breakdown ===
           count    pct
source                 
synthetic  40624  68.71
real       1849

## 5. Data Visualization — Charts & Graphs

### 5.1 Context Class Distribution

In [6]:
# ── Fig 1  Context class distribution (bar + pie) ─────────────────────────────
ctx_counts = full["context"].value_counts()
colors     = [PALETTE.get(c, "#aaaaaa") for c in ctx_counts.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Horizontal bar
bars = axes[0].barh(ctx_counts.index[::-1], ctx_counts.values[::-1], color=colors[::-1])
axes[0].set_xlabel("Sample count")
axes[0].set_title("Context Class Distribution")
for bar, n in zip(bars, ctx_counts.values[::-1]):
    pct = n / len(full) * 100
    axes[0].text(bar.get_width() + 150, bar.get_y() + bar.get_height()/2,
                 f"{n:,}  ({pct:.1f}%)", va="center", fontsize=9)
axes[0].set_xlim(0, ctx_counts.max() * 1.25)

# Pie
wedges, texts, autotexts = axes[1].pie(
    ctx_counts.values, labels=ctx_counts.index,
    colors=colors, autopct="%1.1f%%", startangle=140,
    pctdistance=0.82, textprops={"fontsize": 9}
)
for at in autotexts:
    at.set_fontsize(8)
axes[1].set_title("Context Class Share")

plt.tight_layout()
plt.savefig(CHARTS / "01_context_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → docs/charts/01_context_distribution.png")


Saved → docs/charts/01_context_distribution.png


### 5.2 Severity & Source Breakdown

In [7]:
# ── Fig 2  Severity + Source breakdown ────────────────────────────────────────
sev_counts = full["severity"].value_counts().reindex(["high","medium","low"])
src_counts = full["source"].value_counts()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Severity donut
sev_colors = [SEV_PALETTE[s] for s in sev_counts.index]
wedges, _, autotexts = axes[0].pie(
    sev_counts.values, labels=sev_counts.index, colors=sev_colors,
    autopct="%1.1f%%", startangle=90, pctdistance=0.75,
    wedgeprops=dict(width=0.55))
for at in autotexts:
    at.set_fontsize(9)
axes[0].set_title("Severity Distribution")

# Source bar
axes[1].bar(src_counts.index, src_counts.values, color=["#3498db","#e67e22"], width=0.5)
for i, (label, val) in enumerate(src_counts.items()):
    axes[1].text(i, val + 200, f"{val:,}\n({val/len(full)*100:.1f}%)", ha="center", fontsize=9)
axes[1].set_title("Real vs Synthetic Payloads")
axes[1].set_ylabel("Count")
axes[1].set_ylim(0, src_counts.max() * 1.2)

# Stacked bar: source per context class
ctx_src_piv = full.pivot_table(index="context", columns="source", aggfunc="size", fill_value=0)
ctx_src_piv = ctx_src_piv.loc[ctx_counts.index]
bottom = np.zeros(len(ctx_src_piv))
bar_colors = {"real": "#3498db", "synthetic": "#e67e22"}
for col in ctx_src_piv.columns:
    axes[2].bar(range(len(ctx_src_piv)), ctx_src_piv[col].values,
                bottom=bottom, label=col, color=bar_colors.get(col, "#aaa"))
    bottom += ctx_src_piv[col].values
axes[2].set_xticks(range(len(ctx_src_piv)))
axes[2].set_xticklabels(ctx_src_piv.index, rotation=35, ha="right", fontsize=8)
axes[2].set_title("Source Breakdown per Context")
axes[2].legend()

plt.tight_layout()
plt.savefig(CHARTS / "02_severity_source.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → docs/charts/02_severity_source.png")


Saved → docs/charts/02_severity_source.png


### 5.3 Context × Severity Heatmap

In [8]:
# ── Fig 3  Context × Severity heatmaps ────────────────────────────────────────
ctx_sev_cnt = pd.crosstab(full["context"], full["severity"])
for col in ["low","medium","high"]:
    if col not in ctx_sev_cnt.columns:
        ctx_sev_cnt[col] = 0
ctx_sev_cnt = ctx_sev_cnt[["low","medium","high"]]
ctx_sev_pct = ctx_sev_cnt.div(ctx_sev_cnt.sum(axis=1), axis=0).mul(100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(ctx_sev_cnt, annot=True, fmt="d", cmap="YlOrRd",
            linewidths=0.5, ax=axes[0], cbar_kws={"label": "count"})
axes[0].set_title("Context × Severity  (raw counts)")
axes[0].set_xlabel("Severity"); axes[0].set_ylabel("Context")

sns.heatmap(ctx_sev_pct, annot=True, fmt=".1f", cmap="Blues",
            linewidths=0.5, ax=axes[1], cbar_kws={"label": "row %"},
            vmin=0, vmax=100)
axes[1].set_title("Context × Severity  (row %)")
axes[1].set_xlabel("Severity"); axes[1].set_ylabel("")

plt.tight_layout()
plt.savefig(CHARTS / "03_context_severity_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → docs/charts/03_context_severity_heatmap.png")


Saved → docs/charts/03_context_severity_heatmap.png


### 5.4 Payload Length Distribution

In [9]:
# ── Fig 4  Payload length — overall histogram + KDE ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall histogram
axes[0].hist(full["length"].clip(upper=300), bins=60, color="#3498db", edgecolor="white", alpha=0.85)
axes[0].axvline(full["length"].median(), color="red",   ls="--", lw=1.5, label=f"Median {full['length'].median():.0f}")
axes[0].axvline(full["length"].mean(),   color="orange",ls="--", lw=1.5, label=f"Mean   {full['length'].mean():.1f}")
axes[0].set_title("Payload Length Distribution (clipped @300)")
axes[0].set_xlabel("Length (chars)"); axes[0].set_ylabel("Count")
axes[0].legend()

# KDE per context class (top 6 by count)
top6 = full["context"].value_counts().head(6).index
for ctx in top6:
    sub = full[full["context"] == ctx]["length"].clip(upper=300)
    sub.plot.kde(ax=axes[1], label=ctx, color=PALETTE.get(ctx, "#999"), lw=2)
axes[1].set_title("Length KDE by Context Class  (top 6)")
axes[1].set_xlabel("Length (chars)"); axes[1].set_ylabel("Density")
axes[1].set_xlim(0, 300)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(CHARTS / "04_payload_length_dist.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → docs/charts/04_payload_length_dist.png")


Saved → docs/charts/04_payload_length_dist.png


In [10]:
# ── Fig 5  Box plot: payload length per context class ─────────────────────────
order = full.groupby("context")["length"].median().sort_values().index.tolist()
fig, ax = plt.subplots(figsize=(13, 5))

bp_data = [full[full["context"] == c]["length"].clip(upper=400).values for c in order]
bp = ax.boxplot(bp_data, vert=True, patch_artist=True,
                medianprops=dict(color="white", lw=2),
                flierprops=dict(marker=".", markersize=2, alpha=0.3))
for patch, ctx in zip(bp["boxes"], order):
    patch.set_facecolor(PALETTE.get(ctx, "#aaa"))
    patch.set_alpha(0.85)

ax.set_xticklabels(order, rotation=35, ha="right")
ax.set_title("Payload Length per Context Class (clipped @400)")
ax.set_ylabel("Length (chars)")
ax.set_xlabel("Context Class")

plt.tight_layout()
plt.savefig(CHARTS / "05_length_boxplot_per_context.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → docs/charts/05_length_boxplot_per_context.png")


Saved → docs/charts/05_length_boxplot_per_context.png


### 5.5 Dataset Split Comparison

In [11]:
# ── Fig 6  Train / Val / Test split comparisons ───────────────────────────────
splits   = ["train", "val", "test"]
sp_color = {"train": "#3498db", "val": "#e67e22", "test": "#2ecc71"}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Overall split sizes
sp_counts = full["split"].value_counts().reindex(splits)
axes[0].bar(splits, sp_counts.values, color=[sp_color[s] for s in splits], width=0.5)
for i, (s, n) in enumerate(sp_counts.items()):
    axes[0].text(i, n + 300, f"{n:,}\n({n/len(full)*100:.1f}%)", ha="center", fontsize=9)
axes[0].set_title("Samples per Split")
axes[0].set_ylim(0, sp_counts.max() * 1.2)

# Context distribution per split  (normalised)
for split in splits:
    sub = full[full["split"] == split]["context"].value_counts(normalize=True).reindex(ctx_counts.index, fill_value=0)
    axes[1].plot(range(len(sub)), sub.values, marker="o", label=split, color=sp_color[split], lw=2)
axes[1].set_xticks(range(len(ctx_counts)))
axes[1].set_xticklabels(ctx_counts.index, rotation=35, ha="right", fontsize=8)
axes[1].set_title("Context Mix per Split (normalised)")
axes[1].set_ylabel("Fraction"); axes[1].legend()

# Severity distribution per split (normalised)
for split in splits:
    sub = full[full["split"] == split]["severity"].value_counts(normalize=True).reindex(["low","medium","high"], fill_value=0)
    axes[2].bar(
        [splits.index(split) + i * 3.5 for i in range(3)],
        sub.values, width=0.8, color=[SEV_PALETTE[s] for s in sub.index], alpha=0.85
    )
axes[2].set_xticks([0.5 + i*3.5 for i in range(3)])
axes[2].set_xticklabels(["low", "medium", "high"])
patches = [mpatches.Patch(color=sp_color[s], label=s) for s in splits]
axes[2].legend(handles=patches, fontsize=8)
axes[2].set_title("Severity Mix per Split"); axes[2].set_ylabel("Fraction")

plt.tight_layout()
plt.savefig(CHARTS / "06_split_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → docs/charts/06_split_comparison.png")


Saved → docs/charts/06_split_comparison.png


### 5.6 Boolean Feature Flags & Techniques

In [12]:
# ── Fig 7  Boolean feature flags + technique top-20 ──────────────────────────
flag_cols = [c for c in full.columns if c.startswith("has_")]
flag_counts = full[flag_cols].astype(int).sum().sort_values(ascending=True)
flag_labels = [c.replace("has_","").replace("_"," ") for c in flag_counts.index]

tech_counts = full["technique"].value_counts().head(20)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Flags
bars = axes[0].barh(flag_labels, flag_counts.values,
                    color=plt.cm.tab10.colors[:len(flag_counts)])
for bar, n in zip(bars, flag_counts.values):
    pct = n / len(full) * 100
    axes[0].text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
                 f"{n:,}  ({pct:.1f}%)", va="center", fontsize=8)
axes[0].set_xlim(0, flag_counts.max() * 1.3)
axes[0].set_title("Payload Feature Flag Prevalence")
axes[0].set_xlabel("Count")

# Techniques
axes[1].barh(tech_counts.index[::-1], tech_counts.values[::-1], color="#3498db", alpha=0.85)
for i, (label, val) in enumerate(zip(tech_counts.index[::-1], tech_counts.values[::-1])):
    axes[1].text(val + 50, i, f"{val:,}", va="center", fontsize=8)
axes[1].set_xlim(0, tech_counts.max() * 1.15)
axes[1].set_title("Top 20 Techniques")
axes[1].set_xlabel("Count")

plt.tight_layout()
plt.savefig(CHARTS / "07_flags_and_techniques.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → docs/charts/07_flags_and_techniques.png")


Saved → docs/charts/07_flags_and_techniques.png


### 5.7 Correlation Heatmap & Length Buckets

In [13]:
# ── Fig 8  Correlation heatmap + length bucket distribution ────────────────────
flag_cols = [c for c in full.columns if c.startswith("has_")]
num_df    = full[["length"] + flag_cols].copy()
for c in flag_cols:
    num_df[c] = num_df[c].astype(int)
corr = num_df.corr(numeric_only=True)

bucket_counts = full["payload_len_bucket"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Correlation heatmap
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.4, ax=axes[0],
            xticklabels=[c.replace("has_","") for c in corr.columns],
            yticklabels=[c.replace("has_","") for c in corr.columns])
axes[0].set_title("Feature Correlation Matrix")
axes[0].tick_params(axis="x", rotation=40, labelsize=8)
axes[0].tick_params(axis="y", rotation=0,  labelsize=8)

# Length buckets
axes[1].bar(range(len(bucket_counts)), bucket_counts.values, color="#3498db", alpha=0.85, edgecolor="white")
axes[1].set_xticks(range(len(bucket_counts)))
axes[1].set_xticklabels(bucket_counts.index, rotation=30, ha="right", fontsize=9)
for i, n in enumerate(bucket_counts.values):
    axes[1].text(i, n + 50, f"{n:,}", ha="center", fontsize=8)
axes[1].set_title("Payload Length Buckets")
axes[1].set_ylabel("Count"); axes[1].set_xlabel("Length range (chars)")

plt.tight_layout()
plt.savefig(CHARTS / "08_correlation_and_buckets.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → docs/charts/08_correlation_and_buckets.png")


Saved → docs/charts/08_correlation_and_buckets.png


### 5.8 Severity × Context Grouped Bar & Flag Presence per Context

In [14]:
# ── Fig 9  Grouped bar: high-severity share per context
#           + heatmap: flag presence rate per context ────────────────────────
flag_cols = [c for c in full.columns if c.startswith("has_")]
flag_by_ctx = (
    full.groupby("context")[flag_cols]
    .apply(lambda df: df.astype(int).mean())
    .mul(100)
    .round(1)
)
flag_by_ctx.columns = [c.replace("has_","") for c in flag_by_ctx.columns]

# High-severity % per context
high_pct = (
    full[full["severity"] == "high"]
    .groupby("context").size()
    .div(full.groupby("context").size())
    .mul(100).round(1)
    .reindex(ctx_counts.index, fill_value=0)
    .sort_values(ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Grouped bar (high %)
bars = axes[0].bar(
    range(len(high_pct)), high_pct.values,
    color=[PALETTE.get(c, "#aaa") for c in high_pct.index], alpha=0.9
)
axes[0].set_xticks(range(len(high_pct)))
axes[0].set_xticklabels(high_pct.index, rotation=35, ha="right", fontsize=9)
for bar, v in zip(bars, high_pct.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f"{v:.1f}%", ha="center", fontsize=8)
axes[0].set_title("High-Severity % per Context Class")
axes[0].set_ylabel("% of class labeled high")

# Flag rate heatmap per context
sns.heatmap(flag_by_ctx, annot=True, fmt=".0f", cmap="Greens",
            linewidths=0.4, ax=axes[1], cbar_kws={"label": "prevalence %"})
axes[1].set_title("Feature Flag Prevalence per Context (%)")
axes[1].set_xlabel("Feature"); axes[1].set_ylabel("Context")
axes[1].tick_params(axis="x", rotation=40, labelsize=8)

plt.tight_layout()
plt.savefig(CHARTS / "09_severity_and_flags_per_context.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → docs/charts/09_severity_and_flags_per_context.png")


Saved → docs/charts/09_severity_and_flags_per_context.png


## 6. Statistical Analysis & Key Insights

This section summarises the most important numerical findings from the dataset.

In [15]:
# ── Summary statistics table ──────────────────────────────────────────────────
flag_cols = [c for c in full.columns if c.startswith("has_")]

total = len(full)
n_real = (full["source"] == "real").sum()
n_syn  = (full["source"] == "synthetic").sum()
n_high = (full["severity"] == "high").sum()
n_med  = (full["severity"] == "medium").sum()
n_low  = (full["severity"] == "low").sum()
n_obf  = full["has_obfuscation"].sum()
n_exf  = full["has_cookie_exfil"].sum()
n_tmpl = full["has_template"].sum()
n_dom  = full["has_dom_func"].sum()

dominant_ctx  = full["context"].value_counts().idxmax()
dominant_pct  = full["context"].value_counts(normalize=True).max() * 100
rarest_ctx    = full["context"].value_counts().idxmin()
rarest_pct    = full["context"].value_counts(normalize=True).min() * 100
imbalance_ratio = full["context"].value_counts().max() / full["context"].value_counts().min()

print("=" * 60)
print("  DATASET SUMMARY — RedSentinel XSS Corpus")
print("=" * 60)
print(f"  Total samples          : {total:>10,}")
print(f"    └─ Real payloads      : {n_real:>10,}  ({n_real/total*100:.1f}%)")
print(f"    └─ Synthetic payloads : {n_syn:>10,}  ({n_syn/total*100:.1f}%)")
print(f"  Context classes        :         8")
print(f"  Severity classes       :         3")
print(f"")
print(f"  Severity breakdown")
print(f"    high   : {n_high:>10,}  ({n_high/total*100:.1f}%)")
print(f"    medium : {n_med:>10,}  ({n_med/total*100:.1f}%)")
print(f"    low    : {n_low:>10,}  ({n_low/total*100:.1f}%)")
print(f"")
print(f"  Payload length")
print(f"    mean   : {full['length'].mean():>10.1f} chars")
print(f"    median : {full['length'].median():>10.0f} chars")
print(f"    min    : {full['length'].min():>10.0f} chars")
print(f"    max    : {full['length'].max():>10.0f} chars")
print(f"")
print(f"  Feature prevalence")
print(f"    obfuscated             : {n_obf:>8,}  ({n_obf/total*100:.1f}%)")
print(f"    cookie/storage exfil   : {n_exf:>8,}  ({n_exf/total*100:.1f}%)")
print(f"    template injection     : {n_tmpl:>8,}  ({n_tmpl/total*100:.1f}%)")
print(f"    DOM sink usage         : {n_dom:>8,}  ({n_dom/total*100:.1f}%)")
print(f"")
print(f"  Class balance")
print(f"    dominant class : {dominant_ctx} ({dominant_pct:.1f}%)")
print(f"    rarest class   : {rarest_ctx} ({rarest_pct:.1f}%)")
print(f"    imbalance ratio: {imbalance_ratio:.1f}×")
print("=" * 60)


  DATASET SUMMARY — RedSentinel XSS Corpus
  Total samples          :     59,122
    └─ Real payloads      :     18,498  (31.3%)
    └─ Synthetic payloads :     40,624  (68.7%)
  Context classes        :         8
  Severity classes       :         3

  Severity breakdown
    high   :      9,179  (15.5%)
    medium :     39,856  (67.4%)
    low    :     10,087  (17.1%)

  Payload length
    mean   :       60.9 chars
    median :         54 chars
    min    :          6 chars
    max    :       1504 chars

  Feature prevalence
    obfuscated             :   21,669  (36.7%)
    cookie/storage exfil   :    2,739  (4.6%)
    template injection     :    1,016  (1.7%)
    DOM sink usage         :    2,246  (3.8%)

  Class balance
    dominant class : tag_injection (39.2%)
    rarest class   : generic (1.2%)
    imbalance ratio: 32.3×


In [16]:
# ── Fig 10  Insight dashboard — 4-panel summary chart ─────────────────────────
fig = plt.figure(figsize=(16, 10))
fig.suptitle("RedSentinel XSS Dataset — Insight Dashboard", fontsize=14, fontweight="bold", y=0.98)

ax1 = fig.add_subplot(2, 3, 1)
ax2 = fig.add_subplot(2, 3, 2)
ax3 = fig.add_subplot(2, 3, 3)
ax4 = fig.add_subplot(2, 3, 4)
ax5 = fig.add_subplot(2, 3, 5)
ax6 = fig.add_subplot(2, 3, 6)

# 1. Context bar
ctx_c = full["context"].value_counts()
ax1.barh(ctx_c.index[::-1], ctx_c.values[::-1],
         color=[PALETTE.get(c,"#aaa") for c in ctx_c.index[::-1]], alpha=0.9)
ax1.set_title("Context Classes"); ax1.set_xlabel("Count")

# 2. Severity pie
sev_c = full["severity"].value_counts().reindex(["high","medium","low"])
ax2.pie(sev_c.values, labels=sev_c.index,
        colors=[SEV_PALETTE[s] for s in sev_c.index],
        autopct="%1.1f%%", startangle=90, wedgeprops=dict(width=0.6))
ax2.set_title("Severity Split")

# 3. Source
src_c = full["source"].value_counts()
ax3.bar(src_c.index, src_c.values, color=["#3498db","#e67e22"], width=0.5)
ax3.set_title("Real vs Synthetic"); ax3.set_ylabel("Count")

# 4. Length histogram
ax4.hist(full["length"].clip(upper=250), bins=50, color="#9b59b6", alpha=0.8)
ax4.axvline(full["length"].median(), color="red", ls="--", lw=1.5)
ax4.set_title("Payload Length"); ax4.set_xlabel("chars")

# 5. High-sev % per context
ax5.barh(high_pct.index[::-1], high_pct.values[::-1],
         color=[PALETTE.get(c,"#aaa") for c in high_pct.index[::-1]], alpha=0.85)
ax5.set_title("High-Severity % by Class"); ax5.set_xlabel("%")

# 6. Stacked bar: context × severity
ctx_sev_plot = ctx_sev_cnt.reindex(ctx_c.index)
bottoms = np.zeros(len(ctx_sev_plot))
sev_colors_list = [SEV_PALETTE["low"], SEV_PALETTE["medium"], SEV_PALETTE["high"]]
for col, col_color in zip(["low","medium","high"], sev_colors_list):
    ax6.bar(range(len(ctx_sev_plot)), ctx_sev_plot[col].values,
            bottom=bottoms, color=col_color, label=col, alpha=0.85)
    bottoms += ctx_sev_plot[col].values
ax6.set_xticks(range(len(ctx_sev_plot)))
ax6.set_xticklabels(ctx_sev_plot.index, rotation=40, ha="right", fontsize=7)
ax6.legend(fontsize=8); ax6.set_title("Context × Severity (stacked)")

plt.tight_layout()
plt.savefig(CHARTS / "10_insight_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → docs/charts/10_insight_dashboard.png")


Saved → docs/charts/10_insight_dashboard.png


### Key Findings

| # | Finding | Detail |
|---|---------|--------|
| 1 | **Class balance** | `tag_injection` dominates the dataset; precise breakdown visible after running the stats cell above. |
| 2 | **Severity imbalance** | `high` severity is a minority class — cookie exfil / DOM sink patterns drive it. |
| 3 | **Real vs Synthetic ratio** | ~71 % synthetic — ensures long-tail coverage but may introduce distribution shift. |
| 4 | **Payload length spread** | Short payloads (≤ 60 chars) dominate; `template_injection` tends to produce longer payloads. |
| 5 | **High-severity concentration** | `dom_sink` and `script_injection` contribute the highest proportion of `high` severity. |
| 6 | **Obfuscation prevalence** | A significant minority of payloads use entity/hex/unicode obfuscation — tests model robustness. |
| 7 | **Cookie exfil rarity** | Exfil payloads are intentionally sparse in the real split; synthetic generation addresses this. |
| 8 | **Template injection** | Covered by 31 patterns across 8 template engines; distinct length profile vs other classes. |
| 9 | **Train/Val/Test stratification** | All three splits preserve near-identical context and severity distributions (verified in §5.5). |
| 10 | **Flag correlation** | `has_dom_func` and `has_cookie_exfil` are positively correlated — a DOM sink call is often paired with exfil. |

## 7. Interactive Diagrams & Dashboards

Interactive Plotly charts — hover to explore, zoom, pan, and filter by class.

In [19]:
try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import plotly.io as pio
    HAS_PLOTLY = True
    print("Plotly available ✓")
    print("Interactive charts will be saved as HTML to docs/charts/")
except ImportError:
    HAS_PLOTLY = False
    print("Plotly not installed — run: pip install plotly")
    print("Static charts in §5 remain fully functional.")


Plotly available ✓
Interactive charts will be saved as HTML to docs/charts/


In [20]:
if HAS_PLOTLY:
    # ── Interactive 1: Treemap — context hierarchy ─────────────────────────────
    ctx_sev_flat = full.groupby(["context","severity"]).size().reset_index(name="count")
    ctx_sev_flat["severity_label"] = ctx_sev_flat["severity"].str.upper()

    fig_tree = px.treemap(
        ctx_sev_flat,
        path=["context","severity_label"],
        values="count",
        color="context",
        color_discrete_map=PALETTE,
        title="Dataset Treemap — Context → Severity",
    )
    fig_tree.update_traces(textinfo="label+value+percent parent")
    fig_tree.update_layout(height=550, margin=dict(t=50, l=10, r=10, b=10))
    out = CHARTS / "interactive_01_treemap.html"
    fig_tree.write_html(str(out))
    print(f"Saved → {out}")


Saved → /home/moon/projects/ratopaleydai/docs/charts/interactive_01_treemap.html


In [21]:
if HAS_PLOTLY:
    # ── Interactive 2: Sunburst — source → context → severity ─────────────────
    sun_df = full.groupby(["source","context","severity"]).size().reset_index(name="count")
    fig_sun = px.sunburst(
        sun_df, path=["source","context","severity"], values="count",
        color="context", color_discrete_map=PALETTE,
        title="Sunburst: Source → Context → Severity",
    )
    fig_sun.update_layout(height=600, margin=dict(t=50, l=10, r=10, b=10))
    out = CHARTS / "interactive_02_sunburst.html"
    fig_sun.write_html(str(out))
    print(f"Saved → {out}")


Saved → /home/moon/projects/ratopaleydai/docs/charts/interactive_02_sunburst.html


In [22]:
if HAS_PLOTLY:
    # ── Interactive 3: Violin — payload length per context ────────────────────
    violin_df = full[full["length"] <= 400].copy()
    fig_violin = px.violin(
        violin_df, x="context", y="length", color="context",
        color_discrete_map=PALETTE, box=True, points=False,
        title="Payload Length Distribution per Context (violin)",
        labels={"length": "Length (chars)", "context": "Context Class"},
    )
    fig_violin.update_layout(height=500, showlegend=False,
                             xaxis_tickangle=-35,
                             margin=dict(t=50, b=120))
    out = CHARTS / "interactive_03_violin.html"
    fig_violin.write_html(str(out))
    print(f"Saved → {out}")


Saved → /home/moon/projects/ratopaleydai/docs/charts/interactive_03_violin.html


In [23]:
if HAS_PLOTLY:
    # ── Interactive 4: Sankey — Context → Severity flow ───────────────────────
    ctx_list  = list(full["context"].unique())
    sev_list  = ["low","medium","high"]
    all_nodes = ctx_list + sev_list

    sources, targets, values = [], [], []
    for i, ctx in enumerate(ctx_list):
        for j, sev in enumerate(sev_list):
            v = ((full["context"] == ctx) & (full["severity"] == sev)).sum()
            if v > 0:
                sources.append(i)
                targets.append(len(ctx_list) + j)
                values.append(int(v))

    node_colors = (
        [PALETTE.get(c, "#aaaaaa") for c in ctx_list] +
        [SEV_PALETTE["low"], SEV_PALETTE["medium"], SEV_PALETTE["high"]]
    )

    fig_sankey = go.Figure(go.Sankey(
        node=dict(label=all_nodes, color=node_colors, pad=18, thickness=22),
        link=dict(source=sources, target=targets, value=values,
                  color="rgba(150,150,150,0.25)")
    ))
    fig_sankey.update_layout(
        title_text="Sankey: Context → Severity Flow",
        font_size=11, height=520,
        margin=dict(t=50, l=30, r=30, b=20)
    )
    out = CHARTS / "interactive_04_sankey.html"
    fig_sankey.write_html(str(out))
    print(f"Saved → {out}")


Saved → /home/moon/projects/ratopaleydai/docs/charts/interactive_04_sankey.html


In [24]:
if HAS_PLOTLY:
    # ── Interactive 5: Grouped bar — context distribution per split ───────────
    split_ctx = (
        full.groupby(["split","context"])
        .size().reset_index(name="count")
    )
    split_ctx["pct"] = split_ctx.groupby("split")["count"].transform(lambda x: x / x.sum() * 100).round(2)

    fig_bar = px.bar(
        split_ctx, x="context", y="pct", color="split",
        barmode="group",
        title="Context Class Distribution per Split (%)",
        labels={"pct": "% within split", "context": "Context Class"},
        category_orders={"split": ["train","val","test"]},
        color_discrete_map={"train": "#3498db", "val": "#e67e22", "test": "#2ecc71"},
    )
    fig_bar.update_layout(height=480, xaxis_tickangle=-35,
                          margin=dict(t=50, b=120))
    out = CHARTS / "interactive_05_split_bar.html"
    fig_bar.write_html(str(out))
    print(f"Saved → {out}")


Saved → /home/moon/projects/ratopaleydai/docs/charts/interactive_05_split_bar.html


In [25]:
# ── Final: list all saved chart files ─────────────────────────────────────────
print("=" * 55)
print("  Charts saved to docs/charts/")
print("=" * 55)
for f in sorted(CHARTS.glob("*.png")):
    size_kb = f.stat().st_size // 1024
    print(f"  {f.name:<45} {size_kb:>5} KB")
print("=" * 55)
print(f"\n  Total : {len(list(CHARTS.glob('*.png')))} charts")


  Charts saved to docs/charts/
  01_context_distribution.png                     137 KB
  02_severity_source.png                          115 KB
  03_context_severity_heatmap.png                 131 KB
  04_payload_length_dist.png                      154 KB
  05_length_boxplot_per_context.png               109 KB
  06_split_comparison.png                         128 KB
  07_flags_and_techniques.png                     182 KB
  08_correlation_and_buckets.png                  162 KB
  09_severity_and_flags_per_context.png           177 KB
  10_insight_dashboard.png                        226 KB

  Total : 10 charts
